# Step 1: find clear photos before and after Hurricane Florence

**Goal:** pick the river whose box has clear satellite photos both *before* and *soon after* Florence came ashore (Sept 14, 2018).

New words:
- **Tile:** NASA cuts the Earth into squares about 110 km wide. Each photo file covers one square.
- **Fmask:** a stencil that comes with every HLS photo. It marks each pixel as cloud, cloud shadow, water, and so on.
- **Box clear:** the share of our *whole* box that the satellite saw with no cloud or shadow.

In [1]:
import sys, importlib
from pathlib import Path
sys.path.insert(0, str(Path('..') / 'src'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import hls
step1 = importlib.import_module('01_find_clear_photos')  # reuse the script's functions

/Users/co2/Downloads/nc-hls-mapper/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Sign in
Uses the login saved in `~/.netrc` (made once, by me, in the terminal). No password appears here.

In [2]:
hls.login();

## First look: one photo and its cloud stencil
Lumberton box on **Sept 18, 2018**, 4 days after landfall. Left: the photo in normal colors (red, green and blue bands). Right: the Fmask stencil.

In [3]:
bbox = hls.STUDY_AREAS[hls.STUDY_AREA]
g = [x for x in hls.search(bbox, '2018-09-18', '2018-09-18') if '.S30.' in x['umm']['GranuleUR']][0]
print(g['umm']['GranuleUR'])

# Normal-color picture: HLS stores reflectance x 10000, so multiply by 0.0001.
rgb = np.dstack([hls.read_band(hls.band_url(g, b), bbox).values * 0.0001 for b in ('B04', 'B03', 'B02')])
rgb = np.clip(rgb / 0.25, 0, 1)  # brighten: most land reflects under 25% of light

fmask = hls.read_band(hls.band_url(g, 'Fmask'), bbox).values
cats = np.zeros(fmask.shape, int)                                       # 0 = clear land
cats[(fmask & (1 << 5)) > 0] = 1                                          # 1 = water (Fmask's own guess)
cats[(fmask & ((1 << 2) | (1 << 3))) > 0] = 2                             # 2 = shadow / next to cloud
cats[(fmask & (1 << 1)) > 0] = 3                                          # 3 = cloud
cats[fmask == hls.FMASK_NODATA] = 4                                       # 4 = no data

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(rgb); axes[0].set_title('Sept 18, 2018: normal colors')
cmap = ListedColormap(['#d9d9d9', '#2b83ba', '#7b3294', '#ffffff', '#000000'])
im = axes[1].imshow(cats, cmap=cmap, vmin=-0.5, vmax=4.5, interpolation='nearest')
axes[1].set_title('Fmask stencil')
cb = fig.colorbar(im, ax=axes[1], ticks=range(5)); cb.ax.set_yticklabels(['clear land', 'water', 'shadow / near cloud', 'cloud', 'no data'])
for a in axes: a.set_xticks([]); a.set_yticks([])
fig.tight_layout(); fig.savefig('../figures/01_lumberton_sept18_photo_and_clouds.png', dpi=150)
print('box clear: {:.0%}'.format(hls.clear_fraction(hls.read_band(hls.band_url(g, 'Fmask'), bbox), bbox)[0]))

HLS.S30.T17SPU.2018261T154911.v2.0


box clear: 86%


## Count clear photos for all three river boxes
For every photo from Aug 15 to Oct 10, 2018 we read only the stencil inside each box. This takes about 5 minutes.

A photo counts as **good** if at least 80% of the whole box is clear.

In [4]:
tables = [step1.measure_area(name, b, verbose=False) for name, b in hls.STUDY_AREAS.items()]
df = pd.concat(tables, ignore_index=True)
df['date'] = pd.to_datetime(df['date'])
step1.summarize(df)

,river,photos,good before,good after,first good after,days after landfall
0,Lumber River (Lumberton),28,2,2,2018-09-18,4
1,Neuse River (Kinston),17,1,1,2018-10-03,19
2,Cape Fear River (Fayetteville),29,2,3,2018-10-01,17


Photos in the week after landfall:

In [5]:
week = df[(df['date'] >= '2018-09-15') & (df['date'] <= '2018-09-21')]
week[['area', 'date', 'satellite', 'clear']].round(2)

/var/folders/88/mn9rk90938z4bb1jczzk6z040000gn/T/ipykernel_62163/4193463123.py:2: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  week[['area', 'date', 'satellite', 'clear']].round(2)


,area,date,satellite,clear
16,Lumber River (Lumberton),2018-09-17,Landsat,0.00
17,Lumber River (Lumberton),2018-09-18,Sentinel-2,0.86
18,Lumber River (Lumberton),2018-09-21,Sentinel-2,0.28
37,Neuse River (Kinston),2018-09-18,Sentinel-2,0.63
38,Neuse River (Kinston),2018-09-19,Landsat,0.01
61,Cape Fear River (Fayetteville),2018-09-16,Sentinel-2,0.00
62,Cape Fear River (Fayetteville),2018-09-17,Landsat,0.01
63,Cape Fear River (Fayetteville),2018-09-18,Sentinel-2,0.79
64,Cape Fear River (Fayetteville),2018-09-21,Sentinel-2,0.00


In [6]:
step1.plot_timeline(df, '../figures/01_clear_photos_timeline.png');

## What we learned
- Around a hurricane, most photos are cloudy. Only **Sept 18** gave a usable look soon after landfall.
- **Lumber River (Lumberton)** is the only box over 80% clear that day (86%), and it also has clear "before" photos (Aug 25, Sept 6). **This is our study area.**
- NASA's tile cloud % can mislead for a small box: the tile can be cloudy while our box is clear, or the reverse.
- **A mistake I caught:** the first version divided by the piece of the box a tile covered, not the whole box. A tile touching only 2% of the Cape Fear box scored "97% clear". After the fix, Cape Fear on Sept 18 is 79% and Neuse is 63%.